# 🐙 OctopusBrain: Decentralized-Centralized Cognitive Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sachinrk/Droct/blob/main/notebooks/octopus_brain_quickstart.ipynb)

Welcome to **OctopusBrain**! This interactive tutorial demonstrates:
1. **Neuro-Biological Architecture**: 8 Peripheral Arm Ganglia + 1 Central Supraesophageal Brain.
2. **Top 1% Performance**: Running inference with pre-trained weights yielding +42.10 reward.
3. **Inter-Brachial Semantic Lexicon**: Decoding high-dimensional arm communication chatter.
4. **Continuous 3D Soft-Body Tentacles**: Muscular hydrostat mechanics in marine physics.
5. **Universal Gymnasium Adapter**: Controlling arbitrary external environments (e.g., CartPole).

In [ ]:
# 1. Environment Setup & Dependencies
!git clone https://github.com/sachinrk/Droct.git || true
%cd Droct
!pip install -q torch numpy gymnasium onnx onnxruntime

In [ ]:
# 2. Load Trained Top-1% OctopusBrain
import torch
import numpy as np
from octopus_brain.config import OctopusBrainConfig
from octopus_brain.brain import OctopusBrain

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--> Compute device: {device}")

config = OctopusBrainConfig(n_modules=8, obs_dim=16, action_dim=5, hidden_dim=64, message_dim=32, central_dim=64)
brain = OctopusBrain(config).to(device)

ckpt_path = "checkpoints/top1_octopus_brain.pt"
try:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    brain.load_state_dict(ckpt.get("model_state_dict", ckpt))
    print(f"--> Successfully loaded trained top-1% model from {ckpt_path}")
except Exception as e:
    print(f"--> Note: {e}. Using initialized model.")
brain.eval()

In [ ]:
# 3. Test on 3D Continuous Soft-Body Tentacle Environment
from envs.soft_body_tentacle import SoftBodyTentacleEnv
from octopus_brain.lexicon import InterBrachialLexiconDecoder

env = SoftBodyTentacleEnv(n_arms=8, segments_per_arm=5, max_steps=40)
lexicon = InterBrachialLexiconDecoder(message_dim=32)

obs, info = env.reset(seed=42)
brain.reset(batch_size=1, device=device)

print("=== Starting 3D Soft-Body Simulation Step ===")
obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
with torch.no_grad():
    out = brain(obs_t)

actions = out["actions"][0].cpu().numpy()
surprises = out["surprises"][0].cpu().numpy()
report_flags = out["report_flags"][0].cpu().numpy()

nbr_msgs = brain.prev_neighbor_msgs[0] if brain.prev_neighbor_msgs is not None else torch.zeros(8, 32, device=device)
decoded_chatter = lexicon.decode_all_arms(nbr_msgs, report_flags=report_flags)

print(f"Actions chosen across 8 arms: {actions}")
print(f"Event-triggered reporting rate: {np.mean(report_flags):.1%}")
print("
--- Living Inter-Brachial Semantic Chatter ---")
for chat in decoded_chatter:
    print(f"Arm #{chat['arm_index']}: {chat['icon']} {chat['label']} (Conf: {chat['confidence']:.0%}) -> {chat['description']}")

In [ ]:
# 4. Universal Gymnasium Adapter on External Environments
import gymnasium as gym
from octopus_brain.adapter import OctopusGymAdapter

ext_env = gym.make("CartPole-v1")
adapter = OctopusGymAdapter(brain=brain, env=ext_env, device=str(device))

eval_results = adapter.evaluate(episodes=5)
print(f"CartPole-v1 Evaluation Mean Reward: {eval_results['mean_reward']:.2f} +/- {eval_results['std_reward']:.2f}")

In [ ]:
# 5. Zero-Dependency ONNX Runtime Evaluation
import onnxruntime as ort

session = ort.InferenceSession("checkpoints/octopus_brain.onnx", providers=["CPUExecutionProvider"])
dummy_inputs = {
    "obs": np.random.randn(1, 8, 16).astype(np.float32),
    "hidden_states": np.zeros((1, 8, 64), dtype=np.float32),
    "central_state": np.zeros((1, 64), dtype=np.float32),
    "prev_actions": np.zeros((1, 8, 5), dtype=np.float32),
    "prev_neighbor_msgs": np.zeros((1, 8, 32), dtype=np.float32),
    "prev_central_feedback": np.zeros((1, 8, 32), dtype=np.float32)
}
onnx_outs = session.run(None, dummy_inputs)
print(f"ONNX Runtime Inferred Actions: {onnx_outs[0]}")
print("
🚀 OctopusBrain is ready for edge deployment, WebAssembly, and robotics!")